1.1. Реалізувати процедуру завантаження файлів з VHI-індексом для кожної адміністративної одиниці України (urllib). 
Додати мітку часу до імені файлу та реалізувати механізм запобігання повторному завантаженню.

In [10]:
import urllib.request
import os
from datetime import datetime

def download_vhi_data(directory='vhi_data'):
    # Створюємо папку, якщо вона не існує
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Директорію '{directory}' створено.")

    # Базовий URL для України
    base_url = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={}&year1=1981&year2=2024&type=Mean"

    for i in range(1, 28): # ID 1-27 (ID 0 ігноруємо за завданням)
        # Перевіряємо, чи ми вже завантажували дані для цієї області сьогодні
        existing_files = [f for f in os.listdir(directory) if f.startswith(f"vhi_id_{i}_")]
        
        if existing_files:
            print(f"Область {i}: файл уже існує ({existing_files[0]}). Пропускаємо.")
            continue

        url = base_url.format(i)
        now = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"vhi_id_{i}_{now}.csv"
        path = os.path.join(directory, filename)

        try:
            print(f"Завантаження області {i}...", end=" ")
            urllib.request.urlretrieve(url, path)
            print("Готово.")
        except Exception as e:
            print(f"Помилка: {e}")

# Виклик процедури
download_vhi_data()

Область 1: файл уже існує (vhi_id_1_20260417_122259.csv). Пропускаємо.
Область 2: файл уже існує (vhi_id_2_20260417_122303.csv). Пропускаємо.
Область 3: файл уже існує (vhi_id_3_20260417_122304.csv). Пропускаємо.
Область 4: файл уже існує (vhi_id_4_20260417_122307.csv). Пропускаємо.
Область 5: файл уже існує (vhi_id_5_20260417_122308.csv). Пропускаємо.
Область 6: файл уже існує (vhi_id_6_20260417_122309.csv). Пропускаємо.
Область 7: файл уже існує (vhi_id_7_20260417_122310.csv). Пропускаємо.
Область 8: файл уже існує (vhi_id_8_20260417_122311.csv). Пропускаємо.
Область 9: файл уже існує (vhi_id_9_20260417_122311.csv). Пропускаємо.
Область 10: файл уже існує (vhi_id_10_20260417_122312.csv). Пропускаємо.
Область 11: файл уже існує (vhi_id_11_20260417_122313.csv). Пропускаємо.
Область 12: файл уже існує (vhi_id_12_20260417_122314.csv). Пропускаємо.
Область 13: файл уже існує (vhi_id_13_20260417_122315.csv). Пропускаємо.
Область 14: файл уже існує (vhi_id_14_20260417_122316.csv). Пропускає

1.2. Зчитати завантажені файли у pandas dataframe. Здійснити data cleaning (прибрати зайвий текст, заповнити пропуски). 
Реалізувати процедуру зміни індексів (NOAA English -> Ukrainian Alphabet).

In [11]:
import pandas as pd
import glob

def create_cleaned_dataframe(directory='vhi_data'):
    all_files = glob.glob(os.path.join(directory, "*.csv"))
    list_df = []

    # Словник для реіндексації (NOAA ID -> Ukrainian Alpha ID)
    # Згідно зі списком: 24 (Vinnytsya) стає 1, 25 (Volyn) стає 2 і т.д.
    reindex_map = {
        24: 1, 25: 2, 5: 3, 6: 4, 27: 5, 23: 6, 26: 7, 7: 8, 11: 9, 13: 10, 
        14: 11, 15: 12, 16: 13, 17: 14, 18: 15, 19: 16, 21: 17, 22: 18, 
        8: 19, 9: 20, 10: 21, 1: 22, 3: 23, 2: 24, 4: 25, 12: 26, 20: 27
    }

    for file in all_files:
        filename = os.path.basename(file)
        old_id = int(filename.split('_')[2])
        new_id = reindex_map.get(old_id, old_id)

        # Читаємо файл
        df = pd.read_csv(file, index_col=None, header=1, names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI'])
        
        # ВИПРАВЛЕНО ТУТ: додаємо .astype(str)
        df = df[df['Year'].astype(str).str.contains('<') == False]
        
        # Далі як було: перетворюємо все в числа (те, що не число, стане NaN)
        df = df.apply(pd.to_numeric, errors='coerce')
        df = df.dropna()

        df['Area_ID'] = new_id
        list_df.append(df)

    final_df = pd.concat(list_df, ignore_index=True)
    return final_df

df = create_cleaned_dataframe()
print("Дані завантажено та очищено.")
df.head()

Дані завантажено та очищено.


,Year,Week,SMN,SMT,VCI,TCI,VHI,Area_ID


1.3. Реалізувати функцію для отримання ряду VHI для області за вказаний рік.

In [12]:
def get_vhi_series(df, province_id, year):
    result = df[(df['Area_ID'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]
    return result

print("VHI для області 1 (Вінницька) за 2020 рік:")
display(get_vhi_series(df, province_id=1, year=2020).head())

VHI для області 1 (Вінницька) за 2020 рік:


,Week,VHI


1.4. Реалізувати процедуру для формування вибірки: ряд VHI за вказаний діапазон років для вказаних областей.

In [13]:
def get_vhi_range(df, provinces, start_year, end_year):
    # provinces має бути списком, наприклад [1, 5, 10]
    result = df[(df['Area_ID'].isin(provinces)) & 
                (df['Year'] >= start_year) & 
                (df['Year'] <= end_year)]
    return result

print("VHI для областей 1 та 10 за 2015-2017 роки:")
display(get_vhi_range(df, provinces=[1, 10], start_year=2015, end_year=2017).head(10))

VHI для областей 1 та 10 за 2015-2017 роки:


,Year,Week,SMN,SMT,VCI,TCI,VHI,Area_ID


1.5. Реалізувати процедуру пошуку екстремумів (min та max), середнього та медіани для вказаних областей та років.

In [14]:
def get_vhi_statistics(df, province_id, year):
    filtered_df = df[(df['Area_ID'] == province_id) & (df['Year'] == year)]
    
    # Обчислюємо статистику для колонки VHI
    stats = filtered_df['VHI'].agg(['min', 'max', 'mean', 'median'])
    return stats

print(f"Статистика VHI для області 1 за 2020 рік:")
print(get_vhi_statistics(df, province_id=1, year=2020))

Статистика VHI для області 1 за 2020 рік:
min      NaN
max      NaN
mean     NaN
median   NaN
Name: VHI, dtype: float64
